# A small football feature example

Load the pinned Premier League 2024/25 publication, choose corner features and inspect a few rows.
The [loading notebook](01_loader_walkthrough.ipynb) covers the full experiment population.

**Timing:** by default, earlier finished kickoffs are a retrospective availability proxy.
They do not establish when results or statistics were actually completed or published.
No availability timestamps are required.

In [1]:
from pathlib import Path
import pandas as pd
from xdiyo_analytics.data import load_season, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import (
    Stat, ForAgainst, H2H, IsHome, NormalizedStanding, Lag,
    RollingMean, RollingStd, RollingZScore, evaluate_features, league_season_team_counts,
)

project_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')

In [2]:
season = load_season(
    project_root / 'data/xDiyo_data', 'Premier_League_24_25',
    tables=['matches', 'statistics', 'pregame'],
    record_path=project_root / 'experiment/initial_population/selections/Premier_League_24_25.json',
)
selected = select_stats(season, bundles=['attack_all', 'standings'])
history = build_team_history(selected)
team_counts = league_season_team_counts(history)  # Full season, before any split or filtering.
print(f'{len(season.matches):,} matches; {len(history):,} team rows')
team_counts

380 matches; 760 team rows


competition_id  season_id
17              61627        20
Name: team_count, dtype: int64

## Choose named features

`ALL` means full match. `ForAgainst(..., 'both')` produces separate team/opponent columns.
Windows count eligible matches, even when a value is missing; reductions use finite values inside the window.
The Z-score baseline includes the latest eligible observation. Undefined historical summaries stay missing.
Normalized standings use all 20 teams and default missing/invalid positions to zero.
See the [feature guide](../docs/analytics/features.md) for period expansion, nesting and optional EMA.

In [3]:
corners = Stat('ALL', 'Match overview', 'cornerKicks')
expressions = {
    'home': IsHome(),
    'standing': NormalizedStanding(),
    'corners_lag1': Lag(ForAgainst(corners, 'both')),
    'corners_mean5': RollingMean(corners, window=5),
    'corners_std5': RollingStd(corners, window=5),
    'corners_z5': RollingZScore(corners, window=5),
    'h2h_corners_lag1': Lag(H2H(corners)),
}
features = evaluate_features(history, expressions, team_counts=team_counts)
print(features.shape)
features.columns.tolist()

(760, 8)


['home',
 'standing',
 'corners_lag1::team::ALL::Match overview::cornerKicks::value',
 'corners_lag1::opponent::ALL::Match overview::cornerKicks::value',
 'corners_mean5',
 'corners_std5',
 'corners_z5',
 'h2h_corners_lag1']

In [4]:
preview = pd.concat([
    history[['kickoff_at', 'team_name', 'opponent_name']],
    features[['home', 'standing', 'corners_mean5', 'corners_z5', 'h2h_corners_lag1']],
], axis=1)
preview.tail(4)

,kickoff_at,team_name,opponent_name,home,standing,corners_mean5,corners_z5,h2h_corners_lag1
756,2025-05-25 15:00:00+00:00,Brighton & Hove Albion,Tottenham Hotspur,0.0,0.631579,4.6,-0.555368,4.0
757,2025-05-25 15:00:00+00:00,Tottenham Hotspur,Brighton & Hove Albion,1.0,0.157895,3.4,-1.042493,7.0
758,2025-05-25 15:00:00+00:00,Manchester City,Fulham,0.0,0.894737,7.4,-0.874767,8.0
759,2025-05-25 15:00:00+00:00,Fulham,Manchester City,1.0,0.526316,5.8,-0.675140,3.0


## Optional: predict two days before kickoff

The default is `cutoffs=None`. Supply an aligned datetime Series to move the historical boundary earlier.
Cutoffs change historical features; supplied match context such as standings is not reconstructed as of that earlier time.
This season's corner summaries happen to be unchanged by a two-day cutoff.
See [cutoff formats and frozen rounds](../docs/analytics/feature_cutoffs.md).

In [5]:
cutoffs = history['kickoff_at'] - pd.Timedelta(days=2)
earlier_features = evaluate_features(history, expressions, cutoffs=cutoffs, team_counts=team_counts)
pd.DataFrame({
    'kickoff_at': history['kickoff_at'],
    'prediction_at': cutoffs,
    'corners_mean5': earlier_features['corners_mean5'],
}).tail(4)

,kickoff_at,prediction_at,corners_mean5
756,2025-05-25 15:00:00+00:00,2025-05-23 15:00:00+00:00,4.6
757,2025-05-25 15:00:00+00:00,2025-05-23 15:00:00+00:00,3.4
758,2025-05-25 15:00:00+00:00,2025-05-23 15:00:00+00:00,7.4
759,2025-05-25 15:00:00+00:00,2025-05-23 15:00:00+00:00,5.8
